### 管理员身份！无管理员权限则手动取cookie


In [1]:
import requests
import time
import re
import json
import pandas as pd
import uuid
from websocket import create_connection
# ==========================================
# 1. 核心配置区
# ==========================================
# COOKIE = 'cna=WIexINCop1QCAdzEYl5r2INI; dnk=; lgc=; cookie2=1ef899b7fcec50ad0875edd6269ddcb2; _nk_=; cancelledSubSites=empty; t=5bb8e5b44717c3ebd6b19fffa2f6a626; _tb_token_=e3e54d5aaeebe; xlly_s=1; welcomeShownTime=1776075897020; uc1=cookie21=UIHiLt3xSalX&cookie14=UoYZbLAyI7jpCQ%3D%3D; lid=icebreaker%E6%97%97%E8%88%B0%E5%BA%97%3Amax; unb=2212454123853; sgcookie=E100lYZ%2BNb0Dk%2Fx1XYkvZuJq0%2BnCsKYS4rRk3B8q0WVADeiSmylMC4mqJTE42d0%2Fuez9ShR4YeBvXx58OojowlGGC%2FMtR0ngrlAy82T%2BWZ6rF0PoRrMsL0CHdyxW3PB1A%2Bku; csg=b4e22988; sn=icebreaker%E6%97%97%E8%88%B0%E5%BA%97%3Amax; db_base=50b02bb98c3037e7c3d8250f3f6ce510; db_smart=6b2ef51015a15b5881f11d97df505e5f; isg=BHh4l9YYZQclcoh1-4DujUMASSYK4dxrOHzydbLpxLNmzRi3WvGs-47-h8X9nZRD; __YSF_SESSION__={"baseId":"8c3037e7c3d8250f","brandId":"6ca96e7e7b375583","departmentId":"839e97c85da40a0d","smartId":"d98652764cdabb54","databankProjectId":"7eae325ba969c6d3"}; tfstk=gSZjvNqnsIAXcBWA6IWyFO7T3TisoTSUMdMTKRK2BmnxCcw31OX0QSzSB7PsbcyOi5ZswfcY0xlqB5NzfVW0_ElOfRnxSrlZgfi_IW6PTMSUn-miD65FYphrPfieXxdZBLn-bA6VNVRan-mMEuVqzT2cfUxmqAnT60h-Kvm9Dj3OF4HEBcKxWnL-wbDtXchxk0L-3Ac96qHAF8hoBchT6qBS2bDtXfFtXACGNYbjM-6TYYU5x9fUBbt9XuMRsfefehD8vxgj9-EXXhBihqGLHb1Aba7IyR0_jdxs7-U3svFvM1GgV8EYJWCyujUICJz_9O8mUVZmVfEAxpG8cRZSozRvkvijG2ExJG6QpVU7VmrAse2zFj3I4z7lgVobGycnkafu18G46ogJG_la88rxPWCyV50_Woo8D_IC4aKEOwP95LgHfYGFFTTMS4oUETIdC3QikYDzYT6WGF0xEYGFFTTMSqHoU2W5FITG.'

TARGET_NAMES = ["FY26-ib-buyer-Tmall","FY25-ib-buyer-Tmall","24年4月_25年4月萨洛蒙购买人群"] # 你的目标人群包名称


### 目前自动取cookie  vscode需要管理员身份，浏览器不需要，且必须是同一账号！

In [2]:
import rookiepy
import requests

def get_browser_cookie():
    print("🚀 正在通过 rookiepy 尝试静默提取 Edge Cookie...")
    try:
        # 直接调用 edge 方法，rookiepy 会处理复杂的解密和读取逻辑
        # 可以通过 domains 参数过滤，加快速度
        cookies = rookiepy.edge(domains=["tmall.com"])
        
        # 将对象列表转换为 standard cookie 字符串格式
        cookie_parts = []
        for c in cookies:
            cookie_parts.append(f"{c['name']}={c['value']}")
        
        cookie_str = "; ".join(cookie_parts)
        
        if "_tb_token_" not in cookie_str:
            print("⚠️ 提取成功但未发现登录标识，请确保 Edge 中已登录。")
            return ""
            
        print("✅ Edge Cookie 提取成功！")
        return cookie_str
    except Exception as e:
        print(f"❌ 获取失败: {e}")
        return ""

# 使用
COOKIE = get_browser_cookie()

🚀 正在通过 rookiepy 尝试静默提取 Edge Cookie...
✅ Edge Cookie 提取成功！


In [3]:
COOKIE

'arms_uid=8990dee6-044c-44a0-b201-6face151fcea; cna=WIexINCop1QCAdzEYl5r2INI; skupanel-skuoptions-layout-mode=listMode; pnm_cku822=; arms_uid=b46e21b3-5492-4eaf-8331-0484479988f7; xlly_s=1; XSRF-TOKEN=77fe28bc-8351-45a3-9035-bc7df911c38a; XSRF-TOKEN=5ec6d2e8-e4fc-4325-81b6-352235641909; welcomeShownTime=1776995962936; 3rdPartyCookie=1776996117800; _nk_=; _tb_token_=574380365e8fe; cancelledSubSites=empty; cookie2=1613f22622b7b0f46b28b865a7b689de; csg=ac434c7f; dnk=; lgc=; lid=icebreaker%E6%97%97%E8%88%B0%E5%BA%97%3Amax; sgcookie=E100eoQ7Wvnp8c6RWIvUnk%2B4PPz53NIGC0Vb1CI7%2Bzc0lJ3M0RLrCly0Iv6pNRVzV6NhUW3H1oxukX9gAZk80lJTv%2BHN%2BNcoP2zipiGjTjvuKFCFsGninrTFyNyYhPv1ddc%2B; sn=icebreaker%E6%97%97%E8%88%B0%E5%BA%97%3Amax; t=5bb8e5b44717c3ebd6b19fffa2f6a626; uc1=cookie14=UoYZbLlEE%2FLjTA%3D%3D&cookie21=URm48syIZx9a; unb=2212454123853; __YSF_SESSION__={"strategyBaseId":"b3db1d86ccaf0a9e","strategyBrandId":"6ca96e7e7b375583","strategyDepartmentId":"839e97c85da40a0d","strategySmartId":"d98652764

In [4]:

# 提取 Token
token = (re.search(r'_tb_token_=([^;]+)', COOKIE) or [None, ""])[1]

# 基础 Headers 模板
HEADERS = {
    "accept": "application/json, text/plain, */*",
    "origin": "https://strategy.tmall.com",  
    "referer": "https://strategy.tmall.com/", 
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "x-requested-with": "XMLHttpRequest",
    "x-csrf-token": token,
    "cookie": COOKIE
}

BASE_URL = "https://strategy.tmall.com/api/scapi"

# 为了让导出的 Excel 更好看，咱们把英文字段翻译成中文
TAG_DICT = {
    "daas_tag_pred_gender_20200415091417": "性别",
    "daas_tag_pred_age_level_20200415093010": "年龄",
    "common_receive_province_180d": "省份",
    "common_receive_city_level_180d": "城市等级",
    "pref_purchasing_power": "购买力",
    "interest_prefer": "兴趣偏好"
}


In [5]:

# ==========================================
# 2. 核心功能函数
# ==========================================

# [步骤 1] 检索 crowdId (HTTP GET)
def search_crowd_id(name):
    print(f"\n🔍 [步骤1] 正在检索人群包: 【{name}】...")
    h = HEADERS.copy()
    h["x-custom-router"] = "strategy-CustomCrowd" 
    
    params = {"path": "/api/v1/custom/list", "keyword": name.strip(), "page": 1, "pageSize": 50}
    
    try:
        res = requests.get(BASE_URL, headers=h, params=params, timeout=10).json()
        items = (res.get("data") or {}).get("list") or []
        for item in items:
            if name.strip() in (item.get("crowdName") or item.get("name", "")).strip():
                cid = str(item.get("crowdId") or item.get("id"))
                print(f"🎉 成功命中目标！crowdId: {cid}")
                return cid
    except Exception as e: print(f"❌ 检索失败: {e}")
    return None

# [步骤 2] 下发计算指令 (HTTP POST)
def start_perspective(sid, name):
    print(f"⏳ [步骤2] 正在唤醒阿里后台计算引擎...")
    h = HEADERS.copy()
    h["x-custom-router"] = "strategy-CustomCrowd" 
    
    body = {
        "path": "/v2/perspective/start",
        "paramReportType": "CROWD_PERSPECTIVE",
        "crowdId": int(sid),
        "name": name,
        "cycle": "ONCE",
        "contentType": "application/json"
    }
    
    try:
        res = requests.post(BASE_URL, headers=h, json=body, timeout=10).json()
        status = (res.get("data") or {}).get("reportStatus")
        if status == "CREATING" or status == "SUCCESS" or res.get("data", {}).get("reportId"):
            print("✅ 后台透视计算已就绪！等待 5 秒让系统生成数据...")
            time.sleep(5)
            return True
    except Exception as e: print(f"❌ 唤醒计算失败: {e}")
    return False

# [步骤 3 & 4] WebSocket 窃听数据并导出 Excel
def pull_websocket_data_and_export(crowd_id, name):
    print(f"📞 [步骤3] 正在建立 WebSocket 加密通道窃取数据...")
    ws_url = "wss://ws-insight-engine.tmall.com/"
    
    # 构造 WS 专属 Headers
    ws_headers = [
        "Origin: https://insight-engine.tmall.com",
        "User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        f"Cookie: {COOKIE}"
    ]
    
    # 完美复刻你抓出来的“接头暗号”
    rid = str(int(time.time() * 1000)) + "123" # 伪造时间戳 ID
    payload = {
        "method": "/iSheetCrowdService/offline",
        "headers": {"rid": rid, "type": "PULL", "bizType": "STRATEGY"},
        "body": {
            "args": {
                "id": "661",
                "perspectTaskId": crowd_id,
                "bizParam": {
                    "crowdIds": [crowd_id],
                    "tagList": list(TAG_DICT.keys()) # 把我们需要的标签全扔进去
                },
                "appId": "209"
            }
        }
    }

    ws = None
    try:
        # 连接洞察引擎
        ws = create_connection(ws_url, header=ws_headers, timeout=15)
        ws.send(json.dumps(payload))
        print("📨 暗号已发送，等待阿里回传核心数据包...")
        
        # 接收数据
        for _ in range(15): # 监听 15 次，防干扰包
            msg = ws.recv()
            
            # 如果收到的是纯二进制心跳包，直接跳过
            if type(msg) != str: continue
                
            data = json.loads(msg)
            
            # 找到含有真实数据的包裹
            if data.get("headers", {}).get("type") == "DATA" and "results" in data.get("body", {}):
                print("💎 成功截获核心数据包！正在解析导出...")
                
                # 开始解析 JSON
                rows = []
                target_result = data["body"]["results"][0]
                total_cnt = target_result.get("crowdCnt", 0) # 拿总人数
                tag_results = target_result.get("results", {})
                
                for tag_key, tag_data in tag_results.items():
                    tag_cn_name = TAG_DICT.get(tag_key, tag_key)
                    items = tag_data.get("perspectiveItems", [])
                    
                    for item in items:
                        val_name = item.get("tagValueName", "未知")
                        rate = item.get("rate", 0)
                        tgi = item.get("tgi", "-")
                        
                        # 破解阿里的脱敏：反向算出真实人数
                        real_count = int(total_cnt * rate / 100) if rate else 0
                        
                        rows.append({
                            "人群包名称": name,
                            "人群总覆盖数": total_cnt,
                            "标签维度": tag_cn_name,
                            "特征分类": val_name,
                            "占比(%)": round(rate, 2),
                            "精准人数": real_count,
                            "全网TGI": tgi
                        })
                
                # 导出 Excel
                if rows:
                    filename = f"{name}_策略中心画像.xlsx"
                    df = pd.DataFrame(rows)
                    df.to_excel(filename, index=False)
                    print(f"🎆 完美杀青！Excel 已保存至: {filename}")
                ws.close()
                return True
                
        print("⚠️ 通道已关闭，但没有找到数据包，可能是服务器还没算完。")
        ws.close()
    except Exception as e:
        print(f"❌ WebSocket 通信异常: {e}")
        if ws: ws.close()
        
    return False


In [6]:

# ==========================================
# 3. 终极自动化流水线
# ==========================================
if __name__ == "__main__":
    for name in TARGET_NAMES:
        print(f"\n{'='*50}\n🚀 开始自动化流水线: {name}\n{'='*50}")
        
        # 1. 自动搜索 ID
        crowd_id = search_crowd_id(name)
        if not crowd_id: continue
        
        # 2. 唤醒计算
        started = start_perspective(crowd_id, name)
        if not started: continue
        
        # 3. 窃听并导出
        pull_websocket_data_and_export(crowd_id, name)


🚀 开始自动化流水线: FY26-ib-buyer-Tmall

🔍 [步骤1] 正在检索人群包: 【FY26-ib-buyer-Tmall】...

🚀 开始自动化流水线: FY25-ib-buyer-Tmall

🔍 [步骤1] 正在检索人群包: 【FY25-ib-buyer-Tmall】...

🚀 开始自动化流水线: 24年4月_25年4月萨洛蒙购买人群

🔍 [步骤1] 正在检索人群包: 【24年4月_25年4月萨洛蒙购买人群】...
